# 02 - Deep Dive: Image Classification

This notebook explores OCI Vision's image classification in depth. We look
at confidence distributions using matplotlib bar charts, demonstrate
threshold-based filtering, and visualize the ontology relationships that
the model assigns to its labels.

## Setup

In [ ]:
# !pip install oci-vision-ai[notebooks]

from oci_vision.core.client import VisionClient

client = VisionClient(demo=True)
print(f"Client ready (demo={client.is_demo})")

## Run the analysis

In [ ]:
# Classify the demo image
result = client.classify("dog_closeup.jpg")

print(f"Model version: {result.model_version}")
print(f"Total labels:  {len(result.labels)}")
print(f"Top label:     {result.labels[0].name} ({result.labels[0].confidence_pct}%)")

## Explore the results

### Confidence distribution

The OCI Vision classifier returns many labels with varying confidence.
Let's see the full distribution.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Horizontal bar chart of top 15 labels
top15 = result.labels[:15]
names = [l.name for l in reversed(top15)]
confs = [l.confidence_pct for l in reversed(top15)]

fig, ax = plt.subplots(figsize=(9, 6))
colors = ['#E63946' if c >= 95 else '#457B9D' if c >= 80 else '#A8DADC'
          for c in confs]
bars = ax.barh(names, confs, color=colors)
ax.set_xlabel('Confidence (%)')
ax.set_title('Classification Confidence Distribution (Top 15)')
ax.set_xlim(0, 110)
ax.axvline(x=95, color='#E63946', linestyle='--', alpha=0.5, label='>= 95%')
ax.axvline(x=80, color='#457B9D', linestyle='--', alpha=0.5, label='>= 80%')
for bar, score in zip(bars, confs):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{score:.1f}%', va='center', fontsize=8)
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('classification_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

### Threshold filtering

In [ ]:
# above_threshold returns only labels meeting the confidence bar
for threshold in [0.95, 0.90, 0.80, 0.70]:
    filtered = result.above_threshold(threshold)
    names = [l.name for l in filtered]
    print(f"Threshold {threshold:.0%}: {len(filtered)} labels")
    print(f"  -> {', '.join(names[:8])}{'...' if len(names) > 8 else ''}")
    print()

### Ontology relationships

OCI Vision labels come with ontology metadata showing parent-child
relationships. For example, *Dog -> Pet -> Animal*. Let's explore
these from the raw cached response.

In [ ]:
from oci_vision.gallery import get_cached_response

# Load the raw classification response to access ontology_classes
raw = get_cached_response('dog_closeup', 'classification')
ontology = raw.get('ontology_classes', [])

# Find labels that have parents (i.e., are part of a hierarchy)
hierarchical = [o for o in ontology if o['parent_names']]
print(f"Labels with parent relationships: {len(hierarchical)} / {len(ontology)}")
print()

# Build some hierarchy chains
parent_map = {o['name']: o['parent_names'] for o in ontology}

def trace_hierarchy(name, depth=0):
    """Recursively trace a label up to its roots."""
    chain = [name]
    parents = parent_map.get(name, [])
    if parents and depth < 5:
        for p in parents[:1]:  # follow first parent
            chain.extend(trace_hierarchy(p, depth + 1))
    return chain

# Show hierarchies for interesting labels
for label_name in ['Dog', 'Puppy', 'Wolf', 'Cat', 'Vegetation', 'Grass', 'Lawn']:
    if label_name in parent_map:
        chain = trace_hierarchy(label_name)
        print(f"  {' -> '.join(chain)}")

## Visualize

In [ ]:
# Side-by-side: all labels vs high-confidence only
high = result.above_threshold(0.90)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: top 10 all labels
t10 = result.labels[:10]
ax1.barh([l.name for l in reversed(t10)],
         [l.confidence_pct for l in reversed(t10)],
         color='#457B9D')
ax1.set_xlabel('Confidence (%)')
ax1.set_title('All Labels (Top 10)')
ax1.set_xlim(0, 110)

# Right: only >= 90%
ax2.barh([l.name for l in reversed(high)],
         [l.confidence_pct for l in reversed(high)],
         color='#E63946')
ax2.set_xlabel('Confidence (%)')
ax2.set_title('High Confidence Only (>= 90%)')
ax2.set_xlim(0, 110)

plt.tight_layout()
plt.savefig('classification_threshold_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Under the hood

In [ ]:
import json

# Pydantic model -> dict -> JSON
raw_result = result.model_dump()
print(json.dumps(raw_result, indent=2, default=str)[:2000])
print('...')

## Try it yourself

1. **Custom threshold**: Experiment with `above_threshold()` values between
   0.5 and 0.99 to find the sweet spot for your use case.
2. **Compare images**: Run classification on different images and compare
   how the confidence distributions differ.
3. **Ontology exploration**: Trace more label hierarchies to understand
   how OCI Vision organizes its label taxonomy.
4. **Live mode**: Use `VisionClient()` with real OCI credentials to classify
   your own images.